# Tunisian Arabic ASR on TEDxTN

**Research question:** does adapting multilingual Whisper to TEDxTN improve Tunisian Arabic transcription over zero-shot inference? The Hugging Face repository contains metadata and talk URLs, not audio. Obtain audio under applicable terms and place one WAV per talk in `data/audio/` before running inference or training. Audio/model runs are opt-in below; missing assets are reported rather than fabricated.

In [ ]:
from pathlib import Path
import json, os, random, subprocess, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data_utils import inspect_dataset, inspect_audio_directory, read_csv_rows, load_audio_segment
from src.evaluation import error_rates, normalized_for_scoring

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATA_DIR = ROOT / 'data' / 'metadata'
AUDIO_DIR = Path(os.environ.get('TEDXTN_AUDIO_DIR', ROOT / 'data' / 'audio')).expanduser().resolve()
RUN_TRAINING = False
RUN_SMOKE_TEST = True
print('Python:', sys.version.split()[0])
print('Root:', ROOT)
print('GPU check is deferred until the optional model cell.')

In [ ]:
required = [DATA_DIR / f'{split}.csv' for split in ('train', 'dev', 'test')]
if not all(path.is_file() for path in required):
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'inspect_dataset.py'), '--download-metadata'], check=True)

inspection = inspect_dataset(DATA_DIR)
source = {split: read_csv_rows(DATA_DIR / f'{split}.csv') for split in ('train', 'dev', 'test')}
pd.DataFrame([
    {'split': split, 'segments': info['source']['rows'], 'talks': info['source']['talks'],
     'empty transcripts': len(info['source']['empty_text_rows']),
     'median segment seconds': info['source']['duration_seconds']['median']}
    for split, info in inspection['splits'].items()
])

In [ ]:
train_df = pd.DataFrame(source['train'])
train_df['start_time'] = train_df['start_time'].astype(float)
train_df['end_time'] = train_df['end_time'].astype(float)
train_df['duration_seconds'] = train_df['end_time'] - train_df['start_time']
train_df['transcription'] = train_df['transcription'].fillna('')
train_df['character_count'] = train_df['transcription'].str.len()
train_df['word_count'] = train_df['transcription'].str.split().str.len()
print('Train talk IDs:', train_df['audio_filename'].nunique())
print('Blank/whitespace transcripts:', int((train_df.transcription.str.strip() == '').sum()))
print('Repeated exact talk intervals:', int(train_df.duplicated(['audio_filename','start_time','end_time']).sum()))
print('Source split leakage:', inspection['talk_leakage'])
display(train_df[['duration_seconds','character_count','word_count']].describe().round(2))
from collections import Counter
train_tokens = [token for text in train_df.transcription if text.strip() for token in text.split()]
script_profile = Counter()
for token in train_tokens:
    has_arabic = any('\u0600' <= char <= '\u06ff' for char in token)
    has_latin = any('a' <= char.lower() <= 'z' for char in token)
    script_profile['Arabic-containing'] += int(has_arabic)
    script_profile['Latin-containing'] += int(has_latin)
    script_profile['mixed-script'] += int(has_arabic and has_latin)
print('Whitespace-token types:', len(set(train_tokens)), 'script profile (heuristic, not language labels):', dict(script_profile))
display(pd.DataFrame(Counter(train_tokens).most_common(25), columns=['token','frequency']))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
train_df['duration_seconds'].clip(upper=30).hist(bins=30, ax=axes[0])
axes[0].set(title='Segment duration (clipped at 30 s)', xlabel='seconds')
train_df['character_count'].clip(upper=300).hist(bins=30, ax=axes[1])
axes[1].set(title='Transcript length (clipped at 300 chars)', xlabel='characters')
plt.tight_layout()
figures_dir = ROOT / 'results' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / 'asr_metadata_eda.png', dpi=160)

The paper describes spelling variation and Arabic/French/English code-switching. The corpus annotation uses Arabic script for Tunisian words, Roman script for foreign words, and includes disfluencies. Do not treat script as a reliable language label or rewrite Tunisian Arabic into MSA. The hosted CSVs contain no speaker IDs, region labels, or dialect labels. The paper reports 16 kHz WAV, but this metadata-only inspection cannot verify local audio sample rates or corruption.

In [ ]:
audio_audit = inspect_audio_directory(AUDIO_DIR, source)
audio_audit_path = ROOT / 'results' / 'metrics' / 'audio_inventory.json'
audio_audit_path.parent.mkdir(parents=True, exist_ok=True)
audio_audit_path.write_text(json.dumps(audio_audit, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"Local audio coverage: {audio_audit['present_readable_files']}/{audio_audit['expected_talks']} WAV files")
print('Sample-rate counts:', audio_audit['sample_rate_counts'])
print('Missing files:', audio_audit['missing_files'])
print('Invalid files:', audio_audit['invalid_files'])
print('Out-of-bounds intervals:', len(audio_audit['out_of_bounds_segments']))
print('Readiness by split:', {split: details['ready'] for split, details in audio_audit['splits'].items()})
audio_df = pd.DataFrame(audio_audit['files'])
if not audio_df.empty:
    display(audio_df.duration_seconds.describe().to_frame().T.round(2))
    audio_df.duration_seconds.hist(bins=25, figsize=(7,3), title='Available talk duration (seconds)')
    plt.tight_layout()
TEST_AUDIO_READY = audio_audit['splits']['test']['ready']
TRAIN_DEV_AUDIO_READY = audio_audit['splits']['train']['ready'] and audio_audit['splits']['dev']['ready']

## Evaluation protocol

Report corpus WER/CER twice: minimally normalized (NFC + whitespace) and consistently normalized (Unicode casefolding, punctuation removal, whitespace; no dialect rewriting). WER is edit distance over whitespace-delimited tokens; CER removes whitespace before character edit distance, so spaces themselves are not errors. Exclude blank references and retain the excluded count in the saved report. Compare zero-shot and adapted hypotheses on the identical test segments. Keep test data out of training and model selection.

In [ ]:
RUN_MODEL_EVALUATION = False
ASR_MODEL_ID = 'openai/whisper-small'
ASR_MODEL_REVISION = '973afd24965f72e36ca33b3055d56a652f456b4d'
if RUN_MODEL_EVALUATION:
    if not TEST_AUDIO_READY:
        raise RuntimeError('ASR test inference requires all test WAVs. See results/metrics/audio_inventory.json for exact missing or invalid files.')
    import torch
    from transformers import pipeline
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    device = 0 if torch.cuda.is_available() else -1
    print('CUDA available:', torch.cuda.is_available(), 'device index:', device)
    asr = pipeline('automatic-speech-recognition', model=ASR_MODEL_ID, revision=ASR_MODEL_REVISION, device=device)
    test_rows = [r for r in source['test'] if r['transcription'].strip() and float(r['end_time']) - float(r['start_time']) <= 30.0]
    excluded_test = len(source['test']) - len(test_rows)
    limit = 8 if RUN_SMOKE_TEST else None
    test_rows = test_rows[:limit] if limit else test_rows
    references, predictions = [], []
    baseline_seconds = 0.0
    for row in test_rows:
        audio = load_audio_segment(AUDIO_DIR, row['audio_filename'], float(row['start_time']), float(row['end_time']))
        started = time.perf_counter()
        result = asr({'raw': audio, 'sampling_rate': 16000}, generate_kwargs={'language':'ar','task':'transcribe'})
        baseline_seconds += time.perf_counter() - started
        references.append(row['transcription'])
        predictions.append(result['text'])
    del asr
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    results = {
        'minimal': error_rates(references, predictions),
        'normalized': error_rates([normalized_for_scoring(x) for x in references],
                                  [normalized_for_scoring(x) for x in predictions]),
        'n_examples': len(references), 'excluded_test_segments': excluded_test,
        'model': ASR_MODEL_ID, 'model_revision': ASR_MODEL_REVISION,
        'task': 'zero-shot transcribe', 'inference_seconds': baseline_seconds,
    }
    baseline_test_rows = test_rows
    baseline_predictions = predictions
    baseline_path = ROOT / 'results' / 'metrics' / 'asr_baseline_results.json'
    baseline_path.parent.mkdir(parents=True, exist_ok=True)
    baseline_path.write_text(json.dumps({**results, 'status': 'smoke_test' if RUN_SMOKE_TEST else 'full_test',
        'seed': SEED, 'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'torch_version': torch.__version__, 'transformers_version': __import__('transformers').__version__},
        ensure_ascii=False, indent=2), encoding='utf-8')
    examples_path = ROOT / 'results' / 'qualitative_examples' / 'asr_baseline_predictions.csv'
    examples_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'audio_filename':[r['audio_filename'] for r in test_rows],
        'start_time':[r['start_time'] for r in test_rows], 'end_time':[r['end_time'] for r in test_rows],
        'reference':references, 'prediction':predictions}).to_csv(examples_path, index=False, encoding='utf-8')
    print(json.dumps(results, ensure_ascii=False, indent=2))
else:
    print('Inference skipped. Set RUN_MODEL_EVALUATION=True only after audio and model dependencies are ready.')

In [ ]:
# Opt-in adaptation. The smoke configuration is a pipeline check, not a reported experiment.
if RUN_TRAINING:
    if not (TRAIN_DEV_AUDIO_READY and TEST_AUDIO_READY):
        raise RuntimeError('Fine-tuning/evaluation needs train, dev, and test WAVs. See results/metrics/audio_inventory.json.')
    import torch
    from datasets import Dataset
    from transformers import (WhisperForConditionalGeneration, WhisperProcessor,
        Seq2SeqTrainer, Seq2SeqTrainingArguments)
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    processor = WhisperProcessor.from_pretrained(ASR_MODEL_ID, revision=ASR_MODEL_REVISION, language='ar', task='transcribe')
    model = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL_ID, revision=ASR_MODEL_REVISION)
    model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language='ar', task='transcribe')
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    MAX_WHISPER_SECONDS = 30.0
    def select_whisper_rows(rows):
        nonempty = [r for r in rows if r['transcription'].strip()]
        kept = [r for r in nonempty if float(r['end_time']) - float(r['start_time']) <= MAX_WHISPER_SECONDS]
        return kept, len(rows) - len(kept)
    train_rows, excluded_train = select_whisper_rows(source['train'])
    dev_rows, excluded_dev = select_whisper_rows(source['dev'])
    print('Excluded blank or >30s train/dev segments:', excluded_train, excluded_dev)
    cap_train, cap_dev = (16, 8) if RUN_SMOKE_TEST else (None, None)
    train_rows = train_rows[:cap_train] if cap_train else train_rows
    dev_rows = dev_rows[:cap_dev] if cap_dev else dev_rows
    def encode_rows(rows):
        encoded = []
        for row in rows:
            waveform = load_audio_segment(AUDIO_DIR, row['audio_filename'], float(row['start_time']), float(row['end_time']))
            features = processor.feature_extractor(waveform, sampling_rate=16000).input_features[0]
            labels = processor.tokenizer(row['transcription']).input_ids
            encoded.append({'input_features': features, 'labels': labels})
        return Dataset.from_list(encoded)
    train_ds, dev_ds = encode_rows(train_rows), encode_rows(dev_rows)
    class SpeechSeq2SeqCollator:
        def __init__(self, processor):
            self.processor = processor
        def __call__(self, features):
            audio_features = [{'input_features': item['input_features']} for item in features]
            batch = self.processor.feature_extractor.pad(audio_features, return_tensors='pt')
            label_features = [{'input_ids': item['labels']} for item in features]
            labels = self.processor.tokenizer.pad(label_features, return_tensors='pt')
            batch['labels'] = labels['input_ids'].masked_fill(labels['attention_mask'].ne(1), -100)
            if (batch['labels'][:, 0] == self.processor.tokenizer.bos_token_id).all():
                batch['labels'] = batch['labels'][:, 1:]
            return batch
    collator = SpeechSeq2SeqCollator(processor)
    def compute_metrics(prediction):
        pred_ids = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
        pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)
        label_ids = prediction.label_ids.copy()
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        ref_text = processor.batch_decode(label_ids, skip_special_tokens=True)
        minimal = error_rates(ref_text, pred_text)
        normalized = error_rates([normalized_for_scoring(x) for x in ref_text],
                                 [normalized_for_scoring(x) for x in pred_text])
        return {'wer_minimal': minimal['wer'], 'cer_minimal': minimal['cer'],
                'wer_normalized': normalized['wer'], 'cer_normalized': normalized['cer']}
    args = Seq2SeqTrainingArguments(
        output_dir=str(ROOT / 'results' / 'models' / 'asr-whisper-small'),
        per_device_train_batch_size=1, per_device_eval_batch_size=1,
        gradient_accumulation_steps=8, learning_rate=1e-5,
        num_train_epochs=1 if RUN_SMOKE_TEST else 5,
        eval_strategy='epoch', save_strategy='epoch', logging_steps=5,
        predict_with_generate=True, generation_max_length=225,
        fp16=torch.cuda.is_available(), gradient_checkpointing=True,
        load_best_model_at_end=True, metric_for_best_model='eval_wer_normalized',
        greater_is_better=False, report_to='none', save_total_limit=2,
        seed=SEED, data_seed=SEED, remove_unused_columns=False
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
        eval_dataset=dev_ds, data_collator=collator, compute_metrics=compute_metrics,
        processing_class=processor)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    training_output = trainer.train()
    trainer.save_model(str(ROOT / 'results' / 'models' / 'asr-best'))
    processor.save_pretrained(str(ROOT / 'results' / 'models' / 'asr-best'))
    test_rows, excluded_test = select_whisper_rows(source['test'])
    print('Excluded blank or >30s test segments:', excluded_test)
    test_rows = test_rows[:8] if RUN_SMOKE_TEST else test_rows
    test_ds = encode_rows(test_rows)
    adapted_output = trainer.predict(test_ds, metric_key_prefix='test')
    adapted_ids = adapted_output.predictions[0] if isinstance(adapted_output.predictions, tuple) else adapted_output.predictions
    adapted_predictions = processor.batch_decode(adapted_ids, skip_special_tokens=True)
    adapted_test_metrics = adapted_output.metrics
    peak_gpu_memory_bytes = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
    training_log_history = trainer.state.log_history
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    trainer.model.to('cpu')
    del trainer, model, adapted_output, test_ds
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    expected_test_ids = [(r['audio_filename'], r['start_time'], r['end_time']) for r in test_rows]
    saved_test_ids = [(r['audio_filename'], r['start_time'], r['end_time']) for r in globals().get('baseline_test_rows', [])]
    if 'baseline_predictions' not in globals() or saved_test_ids != expected_test_ids:
        from transformers import pipeline
        base_pipe = pipeline('automatic-speech-recognition', model=ASR_MODEL_ID, revision=ASR_MODEL_REVISION,
            device=0 if torch.cuda.is_available() else -1)
        baseline_predictions = []
        for row in test_rows:
            waveform = load_audio_segment(AUDIO_DIR, row['audio_filename'], float(row['start_time']), float(row['end_time']))
            baseline_predictions.append(base_pipe({'raw': waveform, 'sampling_rate': 16000},
                generate_kwargs={'language':'ar','task':'transcribe'})['text'])
        del base_pipe
    references = [row['transcription'] for row in test_rows]
    def metric_bundle(hypotheses):
        return {'minimal': error_rates(references, hypotheses),
                'normalized': error_rates([normalized_for_scoring(x) for x in references],
                                          [normalized_for_scoring(x) for x in hypotheses])}
    experiment = {'status': 'smoke_test' if RUN_SMOKE_TEST else 'full_test',
        'n_test_segments': len(test_rows), 'excluded_train_segments': excluded_train,
        'excluded_dev_segments': excluded_dev, 'excluded_test_segments': excluded_test,
        'model_revision': ASR_MODEL_REVISION,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'python_version': sys.version.split()[0], 'torch_version': torch.__version__,
        'transformers_version': __import__('transformers').__version__,
        'parameter_count': parameter_count, 'peak_gpu_memory_bytes': peak_gpu_memory_bytes,
        'training_metrics': training_output.metrics,
        'training': {'epochs': 1 if RUN_SMOKE_TEST else 5, 'batch_size': 1,
                     'gradient_accumulation_steps': 8, 'learning_rate': 1e-5},
        'baseline': metric_bundle(baseline_predictions),
        'adapted': metric_bundle(adapted_predictions),
        'trainer_test_metrics': adapted_test_metrics,
        'training_log_history': training_log_history}
    metrics_dir = ROOT / 'results' / 'metrics'
    examples_dir = ROOT / 'results' / 'qualitative_examples'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    examples_dir.mkdir(parents=True, exist_ok=True)
    (metrics_dir / 'asr_results.json').write_text(json.dumps(experiment, ensure_ascii=False, indent=2), encoding='utf-8')
    figures_dir = ROOT / 'results' / 'figures'
    figures_dir.mkdir(parents=True, exist_ok=True)
    history_df = pd.DataFrame(training_log_history)
    if 'loss' in history_df or 'eval_loss' in history_df:
        ax = history_df.plot(x='epoch', y=[column for column in ('loss','eval_loss') if column in history_df], marker='o', figsize=(7,4))
        ax.set_ylabel('loss')
        ax.figure.tight_layout()
        ax.figure.savefig(figures_dir / 'asr_training_curves.png', dpi=160)
    pd.DataFrame({'audio_filename':[r['audio_filename'] for r in test_rows],
        'start_time':[r['start_time'] for r in test_rows], 'end_time':[r['end_time'] for r in test_rows],
        'reference':references, 'baseline':baseline_predictions,
        'adapted':adapted_predictions}).to_csv(examples_dir / 'asr_test_predictions.csv', index=False, encoding='utf-8')
    print(json.dumps(experiment, ensure_ascii=False, indent=2))
else:
    print('Training skipped by default. Review GPU memory and licensed audio before enabling.')

## Interpretation and limitations

Whisper's Arabic language option is not a guarantee of Tunisian dialect quality. The paper's fine-tuned results are contextual baselines, not this notebook's results. Record failed examples as well as successes, document model/package/device/configuration, and run test evaluation only after model selection is complete. Reference: Bougares et al. (2025), [TEDxTN](https://aclanthology.org/2025.arabicnlp-main.22/).